[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/078.%20The%20Fleet%20Sizing%20and%20Composition%20Problem/P78-Tier-1_executed.ipynb)

---


# 78. The Fleet Sizing and Composition Problem
## Tier 1: Mathematical Formulation and Exhaustive Search

### Purpose and inputs

This notebook chooses how many vehicles of each type to assign to each route. The objective is to minimize the modeled fleet cost while meeting each route's daily cargo demand and minimum service level. The inputs are vehicle capacities, acquisition and operating costs, service rates, and route demand and request-arrival rates. The outputs are a feasible fleet assignment, its cost, and route-level service and demand checks.

The solution method is exhaustive search: it enumerates the allowed integer fleet assignments, rejects those that violate a constraint, and selects the least-cost feasible assignment. This method gives an optimum within the stated search bounds for this small example; it is not a mixed-integer programming solver.

### Modeling assumptions and scope

- Requests on each route follow a Poisson arrival process with rate $\\lambda$ requests per day.
- Service times are modeled as exponential. Erlang C applies exactly to a queue with identical, interchangeable servers. Because the example includes vehicle types with different service rates, the queue calculation uses their fleet-size-weighted average rate as a homogeneous-server approximation. The result is therefore an approximation for a heterogeneous fleet.
- Each assigned vehicle contributes its full stated cargo capacity to that route's daily demand capacity. Vehicles are assigned to one route and are not shared between routes.
- For each vehicle type and route, the search allows an integer count from zero through the specified bound. The bound applies to each type on each route, not to the route's total fleet size.
- The objective adds one-time acquisition cost and one operating-cost period for each assigned vehicle. This is a simplified cost measure; no planning horizon, discounting, vehicle resale, or replacement is modeled.
- Demand and service-level requirements are enforced separately for every route. Other real-world details, such as travel-time variability, vehicle availability, maintenance, and time-dependent demand, are outside this model.

### Mathematical model

Let $i$ index vehicle types and $j$ index routes. The integer decision variable $x_{ij}$ is the number of vehicles of type $i$ assigned to route $j$. Let $c_i$ be the vehicle's cargo capacity, $a_i$ its acquisition cost, $o_{ij}$ its operating cost on route $j$, and $\mu_i$ its service rate. Route $j$ has daily cargo demand $D_j$, request-arrival rate $\lambda_j$, and maximum acceptable delay probability $\alpha_j$.

The fleet must cover cargo demand:

$$\sum_i c_i x_{ij} \geq D_j \qquad \text{for every route } j.$$

The decision variables are nonnegative integers and are bounded in the search by $0 \leq x_{ij} \leq U$, where $U$ is `max_vehicles_per_route` in the code. For each route, the number of servers and the average service rate per server are

$$s_j = \sum_i x_{ij}, \qquad \bar{\mu}_j = \frac{\sum_i \mu_i x_{ij}}{s_j} \quad (s_j > 0).$$

The queue approximation requires $\lambda_j < s_j\bar{\mu}_j$. Define the offered load $r_j = \lambda_j/\bar{\mu}_j$. The Erlang-C probability that an arriving request must wait is

$$C(s_j,r_j) = \frac{\dfrac{r_j^{s_j}}{s_j!}\dfrac{s_j}{s_j-r_j}}{\displaystyle\sum_{k=0}^{s_j-1}\frac{r_j^k}{k!} + \dfrac{r_j^{s_j}}{s_j!}\dfrac{s_j}{s_j-r_j}}.$$

The service-level constraint is $1-C(s_j,r_j) \geq 1-\alpha_j$, or equivalently $C(s_j,r_j) \leq \alpha_j$, for every route. The modeled total cost is

$$\min_{x}\; \sum_j\sum_i x_{ij}(a_i+o_{ij}).$$

Here, `capacity`, `acquisition_cost`, `operating_costs`, and `service_rate` hold the vehicle parameters; `demand_units`, `arrival_rate`, and `max_delay_prob` hold the route parameters. `calculate_service_level` evaluates the queue constraint, `check_demand_satisfaction` checks the cargo constraint, and `calculate_total_cost` evaluates the objective.

### Solution sequence

1. Define the vehicle types and routes.
2. Calculate delay probability using the Erlang-C formula and convert it to a service level.
3. Enumerate integer assignments within the stated bounds.
4. Reject assignments that fail cargo-demand or service-level constraints.
5. Select the feasible assignment with the lowest modeled cost, then inspect its cost components and sensitivity to demand.

For $m$ vehicle types, $n$ routes, and a per-type bound $U$, the unfiltered search space has at most $(U+1)^{mn}$ assignments. Exhaustive enumeration is useful for small instances because each candidate is checked directly, but the number of candidates grows rapidly as the instance or bounds increase.

### Concrete example

The example has three vehicle types and two routes. Small, medium, and large trucks have cargo capacities of 5, 10, and 20 units and acquisition costs of $50,000, $80,000, and $120,000, respectively. The urban route has daily cargo demand of 15 units and 8 requests per day; the rural route has daily cargo demand of 25 units and 4 requests per day. The code below supplies the route-specific operating costs, service rates, and maximum delay probabilities.

In [ ]:
from dataclasses import dataclass
from itertools import product
import math
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class VehicleType:
    """Characteristics of one vehicle type."""

    id: int
    name: str
    capacity: int
    acquisition_cost: float
    operating_costs: Dict[int, float]
    service_rate: float


@dataclass
class Route:
    """Daily cargo demand and request-arrival requirements for a route."""

    id: int
    name: str
    demand_units: int
    arrival_rate: float
    max_delay_prob: float = 0.1


@dataclass
class FleetSolution:
    """Fleet assignment and its evaluated objective and constraints."""

    assignments: Dict[Tuple[int, int], int]
    total_cost: float
    service_levels: Dict[int, float]
    unmet_demand: Dict[int, int]

In [ ]:
def erlang_c_delay_probability(
    arrival_rate: float,
    service_rate: float,
    num_servers: int,
) -> float:
    """Return the Erlang-C probability that an arrival must wait.

    ``service_rate`` is the service rate per homogeneous server. For the
    example's mixed fleet, the caller supplies the fleet-size-weighted
    average service rate, making this a homogeneous-server approximation.
    Rates must use the same time unit.
    """
    if arrival_rate < 0:
        raise ValueError("arrival_rate must be nonnegative")
    if service_rate <= 0:
        raise ValueError("service_rate must be positive")
    if num_servers < 0:
        raise ValueError("num_servers must be nonnegative")
    if num_servers == 0:
        return 1.0

    offered_load = arrival_rate / service_rate

    # Erlang C has no finite steady-state queue when offered load reaches capacity.
    if offered_load >= num_servers:
        return 1.0

    denominator_sum = sum(
        (offered_load ** k) / math.factorial(k)
        for k in range(num_servers)
    )
    final_term = (
        (offered_load ** num_servers)
        / math.factorial(num_servers)
        * num_servers
        / (num_servers - offered_load)
    )

    return final_term / (denominator_sum + final_term)

In [ ]:
def calculate_service_level(
    route: Route,
    vehicle_assignments: List[Tuple[VehicleType, int]],
) -> float:
    """Return one minus the route's Erlang-C delay probability.

    Mixed vehicle types are represented by their fleet-size-weighted
    average service rate, as described in the model assumptions.
    """
    total_servers = sum(count for _, count in vehicle_assignments)
    if total_servers == 0:
        return 0.0
    if any(count < 0 for _, count in vehicle_assignments):
        raise ValueError("Vehicle counts must be nonnegative")

    total_service_rate = sum(
        vehicle.service_rate * count
        for vehicle, count in vehicle_assignments
    )
    average_service_rate = total_service_rate / total_servers
    delay_probability = erlang_c_delay_probability(
        route.arrival_rate,
        average_service_rate,
        total_servers,
    )

    return 1.0 - delay_probability

In [ ]:
def calculate_total_cost(
    assignments: Dict[Tuple[int, int], int],
    vehicles: List[VehicleType],
) -> float:
    """Sum acquisition and one-period operating costs for assigned vehicles."""
    vehicles_by_id = {vehicle.id: vehicle for vehicle in vehicles}
    total_cost = 0.0

    for (vehicle_id, route_id), count in assignments.items():
        vehicle = vehicles_by_id[vehicle_id]
        total_cost += count * vehicle.acquisition_cost
        total_cost += count * vehicle.operating_costs[route_id]

    return total_cost

In [ ]:
def check_demand_satisfaction(
    assignments: Dict[Tuple[int, int], int],
    vehicles: List[VehicleType],
    routes: List[Route],
) -> Dict[int, int]:
    """Return unmet cargo units for each route; zero means demand is met."""
    vehicles_by_id = {vehicle.id: vehicle for vehicle in vehicles}
    unmet_demand = {}

    for route in routes:
        total_capacity = sum(
            count * vehicles_by_id[vehicle_id].capacity
            for (vehicle_id, route_id), count in assignments.items()
            if route_id == route.id
        )
        unmet_demand[route.id] = max(0, route.demand_units - total_capacity)

    return unmet_demand

In [ ]:
def exhaustive_search_optimization(
    vehicles: List[VehicleType],
    routes: List[Route],
    max_vehicles_per_route: int = 5,
) -> FleetSolution:
    """Find the least-cost feasible assignment within the integer search bounds.

    The bound applies independently to each vehicle type on each route.
    Exhaustive search is intended for small instances.
    """
    if max_vehicles_per_route < 0:
        raise ValueError("max_vehicles_per_route must be nonnegative")

    best_solution = None
    best_cost = float("inf")
    route_combinations = []

    # Enumerating each route's integer counts makes the search bounds explicit.
    for route in routes:
        combinations_for_route = []
        for counts in product(
            range(max_vehicles_per_route + 1),
            repeat=len(vehicles),
        ):
            if sum(counts) == 0:
                continue

            assignment = {
                (vehicles[i].id, route.id): count
                for i, count in enumerate(counts)
                if count > 0
            }
            combinations_for_route.append(assignment)

        route_combinations.append(combinations_for_route)

    if any(not combinations_for_route for combinations_for_route in route_combinations):
        return None

    for route_assignments in product(*route_combinations):
        full_assignment = {}
        for route_assignment in route_assignments:
            full_assignment.update(route_assignment)

        unmet_demand = check_demand_satisfaction(
            full_assignment,
            vehicles,
            routes,
        )
        if any(unmet_demand.values()):
            continue

        service_levels = {}
        feasible = True
        for route in routes:
            vehicle_assignments = [
                (vehicle, count)
                for (vehicle_id, route_id), count in full_assignment.items()
                if route_id == route.id
                for vehicle in vehicles
                if vehicle.id == vehicle_id
            ]
            service_level = calculate_service_level(route, vehicle_assignments)
            service_levels[route.id] = service_level

            required_service_level = 1.0 - route.max_delay_prob
            if service_level < required_service_level:
                feasible = False
                break

        if not feasible:
            continue

        total_cost = calculate_total_cost(full_assignment, vehicles)
        tied_cost = math.isclose(
            total_cost,
            best_cost,
            rel_tol=1e-12,
            abs_tol=1e-9,
        )
        if total_cost < best_cost and not tied_cost:
            best_cost = total_cost
            best_solution = FleetSolution(
                assignments=full_assignment,
                total_cost=total_cost,
                service_levels=service_levels,
                unmet_demand=unmet_demand,
            )

    return best_solution

In [ ]:
# Define the vehicle and route data for the example.
vehicles = [
    VehicleType(
        id=1,
        name="Small Truck",
        capacity=5,
        acquisition_cost=50000,
        operating_costs={1: 1000, 2: 1500},
        service_rate=4.0,
    ),
    VehicleType(
        id=2,
        name="Medium Truck",
        capacity=10,
        acquisition_cost=80000,
        operating_costs={1: 1200, 2: 1800},
        service_rate=3.5,
    ),
    VehicleType(
        id=3,
        name="Large Truck",
        capacity=20,
        acquisition_cost=120000,
        operating_costs={1: 1500, 2: 2000},
        service_rate=3.0,
    ),
]

routes = [
    Route(
        id=1,
        name="Urban Route",
        demand_units=15,
        arrival_rate=8.0,
        max_delay_prob=0.1,
    ),
    Route(
        id=2,
        name="Rural Route",
        demand_units=25,
        arrival_rate=4.0,
        max_delay_prob=0.1,
    ),
]

print("Problem setup")
print(f"Vehicle types: {len(vehicles)}")
for vehicle in vehicles:
    print(
        f"  {vehicle.name}: capacity {vehicle.capacity} units, "
        f"acquisition cost ${vehicle.acquisition_cost:,.0f}, "
        f"service rate {vehicle.service_rate:.1f} requests/day"
    )

print(f"\nRoutes: {len(routes)}")
for route in routes:
    print(
        f"  {route.name}: demand {route.demand_units} units/day, "
        f"arrival rate {route.arrival_rate:.1f} requests/day, "
        f"maximum delay probability {route.max_delay_prob:.1%}"
    )

In [ ]:
# The search evaluates each bounded integer assignment against both constraints.
print("Solving the fleet sizing and composition problem")
print("Method: exhaustive search over bounded integer assignments\n")

solution = exhaustive_search_optimization(
    vehicles,
    routes,
    max_vehicles_per_route=3,
)

if solution is None:
    print("No feasible solution was found within the stated search bounds.")
else:
    print("Feasible minimum-cost solution within the search bounds")
    print("\nFleet composition:")
    for (vehicle_id, route_id), count in solution.assignments.items():
        vehicle = next(vehicle for vehicle in vehicles if vehicle.id == vehicle_id)
        route = next(route for route in routes if route.id == route_id)
        print(f"  {count} x {vehicle.name} assigned to {route.name}")

    print(f"\nModeled total cost: ${solution.total_cost:,.2f}")
    print("\nService levels:")
    for route_id, service_level in solution.service_levels.items():
        route = next(route for route in routes if route.id == route_id)
        delay_probability = 1.0 - service_level
        print(
            f"  {route.name}: service level {service_level:.1%}; "
            f"delay probability {delay_probability:.1%} "
            f"(maximum {route.max_delay_prob:.1%})"
        )

    print("\nCargo-demand checks:")
    for route_id, unmet in solution.unmet_demand.items():
        route = next(route for route in routes if route.id == route_id)
        print(f"  {route.name}: {unmet} unmet units")

    assert all(unmet == 0 for unmet in solution.unmet_demand.values())
    assert all(
        solution.service_levels[route.id] >= 1.0 - route.max_delay_prob
        for route in routes
    )

In [ ]:
def analyze_cost_breakdown(
    solution: FleetSolution,
    vehicles: List[VehicleType],
    routes: List[Route],
) -> Dict[str, Dict[str, float]]:
    """Return acquisition and operating costs by vehicle type and route."""
    vehicles_by_id = {vehicle.id: vehicle for vehicle in vehicles}
    routes_by_id = {route.id: route for route in routes}
    acquisition_costs = {}
    operating_costs = {}

    for (vehicle_id, route_id), count in solution.assignments.items():
        vehicle = vehicles_by_id[vehicle_id]
        route = routes_by_id[route_id]
        label = f"{vehicle.name} on {route.name}"
        acquisition_costs[label] = count * vehicle.acquisition_cost
        operating_costs[label] = count * vehicle.operating_costs[route_id]

    return {
        "acquisition": acquisition_costs,
        "operating": operating_costs,
    }


if solution is not None:
    cost_breakdown = analyze_cost_breakdown(solution, vehicles, routes)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    acquisition_labels = list(cost_breakdown["acquisition"])
    acquisition_values = list(cost_breakdown["acquisition"].values())
    if acquisition_values:
        acquisition_colors = plt.cm.Greens(
            np.linspace(0.4, 0.8, len(acquisition_values))
        )
        ax1.pie(
            acquisition_values,
            labels=acquisition_labels,
            autopct="%1.1f%%",
            colors=acquisition_colors,
            startangle=90,
        )
    ax1.set_title("Acquisition cost breakdown")

    operating_labels = list(cost_breakdown["operating"])
    operating_values = list(cost_breakdown["operating"].values())
    if operating_values:
        operating_colors = plt.cm.Blues(
            np.linspace(0.4, 0.8, len(operating_values))
        )
        ax2.pie(
            operating_values,
            labels=operating_labels,
            autopct="%1.1f%%",
            colors=operating_colors,
            startangle=90,
        )
    ax2.set_title("Operating cost breakdown for one period")

    plt.tight_layout()
    plt.show()

    total_acquisition = sum(cost_breakdown["acquisition"].values())
    total_operating = sum(cost_breakdown["operating"].values())
    assert math.isclose(
        total_acquisition + total_operating,
        solution.total_cost,
        rel_tol=1e-12,
        abs_tol=1e-9,
    )

    print("Cost breakdown")
    print("=" * 50)
    print(
        f"Acquisition costs: ${total_acquisition:,.2f} "
        f"({total_acquisition / solution.total_cost:.1%})"
    )
    for label, cost in cost_breakdown["acquisition"].items():
        print(f"  {label}: ${cost:,.2f}")

    print(
        f"\nOperating costs: ${total_operating:,.2f} "
        f"({total_operating / solution.total_cost:.1%})"
    )
    for label, cost in cost_breakdown["operating"].items():
        print(f"  {label}: ${cost:,.2f}")

In [ ]:
def sensitivity_analysis(
    vehicles: List[VehicleType],
    base_routes: List[Route],
) -> None:
    """Re-solve the bounded search for scaled cargo and arrival demand."""
    demand_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2, 1.3]
    results = []

    print("Sensitivity analysis: demand variations")
    print("=" * 68)
    print(
        f"{'Multiplier':<14} {'Total cost':<18} "
        f"{'Fleet size':<14} {'Feasible':<10}"
    )
    print("-" * 68)

    for multiplier in demand_multipliers:
        modified_routes = [
            Route(
                id=route.id,
                name=route.name,
                demand_units=int(route.demand_units * multiplier),
                arrival_rate=route.arrival_rate * multiplier,
                max_delay_prob=route.max_delay_prob,
            )
            for route in base_routes
        ]

        candidate = exhaustive_search_optimization(
            vehicles,
            modified_routes,
            max_vehicles_per_route=4,
        )
        if candidate is None:
            results.append(
                {
                    "multiplier": multiplier,
                    "cost": None,
                    "fleet_size": 0,
                    "feasible": False,
                }
            )
            print(f"{multiplier:<14.1f} {'—':<18} {0:<14} No")
        else:
            fleet_size = sum(candidate.assignments.values())
            results.append(
                {
                    "multiplier": multiplier,
                    "cost": candidate.total_cost,
                    "fleet_size": fleet_size,
                    "feasible": True,
                }
            )
            print(
                f"{multiplier:<14.1f} ${candidate.total_cost:<17,.2f} "
                f"{fleet_size:<14} Yes"
            )

    feasible_results = [result for result in results if result["feasible"]]
    if not feasible_results:
        print("\nNo feasible cases are available to plot.")
        return

    multipliers = [result["multiplier"] for result in feasible_results]
    costs = [result["cost"] for result in feasible_results]
    fleet_sizes = [result["fleet_size"] for result in feasible_results]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    ax1.plot(multipliers, costs, "o-", linewidth=2, markersize=8)
    ax1.set_xlabel("Demand multiplier")
    ax1.set_ylabel("Modeled total cost ($)")
    ax1.set_title("Cost sensitivity to demand")
    ax1.grid(True, alpha=0.3)
    ax1.ticklabel_format(style="plain", axis="y")

    ax2.plot(multipliers, fleet_sizes, "s-", linewidth=2, markersize=8)
    ax2.set_xlabel("Demand multiplier")
    ax2.set_ylabel("Number of assigned vehicles")
    ax2.set_title("Fleet-size sensitivity to demand")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


sensitivity_analysis(vehicles, routes)

In [ ]:
def service_level_analysis(
    vehicles: List[VehicleType],
    routes: List[Route],
) -> None:
    """Show how the modeled service level changes with a route's fleet size."""
    medium_truck = next(vehicle for vehicle in vehicles if vehicle.id == 2)
    urban_route = next(route for route in routes if route.id == 1)
    truck_counts = list(range(1, 6))
    service_levels = []
    delay_probabilities = []

    print("Service-level analysis")
    print("=" * 58)
    print(f"Vehicle: {medium_truck.name}; route: {urban_route.name}")
    print(
        f"{'Vehicles':<12} {'Service level':<18} "
        f"{'Delay probability':<22} {'Capacity (units/day)':<22}"
    )
    print("-" * 74)

    for count in truck_counts:
        service_level = calculate_service_level(
            urban_route,
            [(medium_truck, count)],
        )
        delay_probability = 1.0 - service_level
        capacity = count * medium_truck.capacity
        service_levels.append(service_level)
        delay_probabilities.append(delay_probability)
        print(
            f"{count:<12} {service_level:<18.1%} "
            f"{delay_probability:<22.1%} {capacity:<22}"
        )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    ax1.plot(truck_counts, service_levels, "o-", linewidth=2, markersize=8)
    ax1.axhline(
        y=1.0 - urban_route.max_delay_prob,
        color="red",
        linestyle="--",
        label=f"Required service level ({1.0 - urban_route.max_delay_prob:.0%})",
    )
    ax1.set_xlabel("Number of medium trucks")
    ax1.set_ylabel("Service level")
    ax1.set_title("Service level versus fleet size")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)

    ax2.plot(
        truck_counts,
        delay_probabilities,
        "s-",
        linewidth=2,
        markersize=8,
    )
    ax2.axhline(
        y=urban_route.max_delay_prob,
        color="red",
        linestyle="--",
        label=f"Maximum allowed ({urban_route.max_delay_prob:.0%})",
    )
    ax2.set_xlabel("Number of medium trucks")
    ax2.set_ylabel("Delay probability")
    ax2.set_title("Delay probability versus fleet size")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1)

    plt.tight_layout()
    plt.show()


service_level_analysis(vehicles, routes)

## Interpreting the method and its limits

Exhaustive search is transparent: every assignment inside the specified integer bounds is evaluated, and the selected assignment has the lowest modeled cost among those satisfying both route-level constraints. If more than one assignment has a cost equal within numerical tolerance, the search retains the first one encountered; the returned assignment is not necessarily the only optimum.

The guarantee is limited to the enumerated search space. Increasing the per-type bound or the number of vehicle types and routes increases the number of assignments rapidly. The Erlang-C service check is also a homogeneous-server approximation for mixed vehicle types, and the cost objective combines acquisition cost with one operating-cost period. These assumptions should be considered when interpreting the solution as a planning result.

The cost breakdown separates the two modeled cost components. The demand sensitivity analysis scales both cargo demand and request-arrival rate, then solves the same bounded search for each multiplier. A case marked infeasible means no assignment within that experiment's search bounds satisfies the model constraints; it does not establish infeasibility beyond those bounds. The service-level analysis isolates the queueing calculation by varying the number of medium trucks on the urban route.

In [ ]:
print("Tier 1 summary: fleet sizing and composition")
print("=" * 58)
print("Method: bounded exhaustive search over integer fleet assignments")
print("Queueing model: Erlang C with a weighted-average service-rate approximation")

if solution is None:
    print("No feasible assignment was found within the main search bounds.")
else:
    print(f"Minimum modeled cost within the search bounds: ${solution.total_cost:,.2f}")
    print(f"Assigned vehicles: {sum(solution.assignments.values())}")
    print(
        "Cargo demand satisfied on all routes: "
        f"{all(unmet == 0 for unmet in solution.unmet_demand.values())}"
    )
    service_constraints_met = all(
        solution.service_levels[route.id] >= 1.0 - route.max_delay_prob
        for route in routes
    )
    print(f"Service-level constraints met on all routes: {service_constraints_met}")

print("\nInterpretation")
print("The optimality claim applies only to the assignments within the search bounds.")
print("Queueing and cost assumptions determine how closely the model represents operations.")